<a href="https://colab.research.google.com/github/AshokGit544/Enterprise_RAG_Document_Intelligence_Platform/blob/main/Enterprise_RAG_Document_Intelligence_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip -q install pandas numpy scikit-learn

In [4]:
from pathlib import Path
from datetime import datetime
import random
import json
import re

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

BASE_PATH = Path('/content/enterprise_rag_document_intelligence_platform')
RAW_PATH = BASE_PATH / 'data' / 'raw_documents'
PROCESSED_PATH = BASE_PATH / 'data' / 'processed_documents'
CHUNK_PATH = BASE_PATH / 'data' / 'chunked_documents'
VECTOR_PATH = BASE_PATH / 'data' / 'vector_ready'
RETRIEVAL_PATH = BASE_PATH / 'data' / 'retrieval_outputs'
OUTPUT_PATH = BASE_PATH / 'outputs'
CONFIG_PATH = BASE_PATH / 'config'

for p in [RAW_PATH, PROCESSED_PATH, CHUNK_PATH, VECTOR_PATH, RETRIEVAL_PATH, OUTPUT_PATH, CONFIG_PATH]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(42)
np.random.seed(42)

print('Project folders created successfully.')
print(BASE_PATH)

Project folders created successfully.
/content/enterprise_rag_document_intelligence_platform


In [5]:
project_schema = {
    'document_types': [
        'quality_report',
        'supplier_invoice',
        'maintenance_record',
        'engineering_change_notice'
    ],
    'domains': [
        'quality',
        'supplier',
        'maintenance',
        'engineering'
    ],
    'fields_to_extract': [
        'document_id',
        'plant_code',
        'supplier_name',
        'invoice_amount',
        'issue_type',
        'equipment_id',
        'change_order_id',
        'document_date'
    ],
    'rag_pipeline_steps': [
        'document_generation',
        'document_preprocessing',
        'document_classification',
        'structured_extraction',
        'chunking',
        'metadata_enrichment',
        'retrieval_indexing',
        'top_k_retrieval',
        'grounded_response_generation',
        'evaluation'
    ]
}

with open(CONFIG_PATH / 'project_schema.json', 'w') as f:
    json.dump(project_schema, f, indent=2)

print(json.dumps(project_schema, indent=2))

{
  "document_types": [
    "quality_report",
    "supplier_invoice",
    "maintenance_record",
    "engineering_change_notice"
  ],
  "domains": [
    "quality",
    "supplier",
    "maintenance",
    "engineering"
  ],
  "fields_to_extract": [
    "document_id",
    "plant_code",
    "supplier_name",
    "invoice_amount",
    "issue_type",
    "equipment_id",
    "change_order_id",
    "document_date"
  ],
  "rag_pipeline_steps": [
    "document_generation",
    "document_preprocessing",
    "document_classification",
    "structured_extraction",
    "chunking",
    "metadata_enrichment",
    "retrieval_indexing",
    "top_k_retrieval",
    "grounded_response_generation",
    "evaluation"
  ]
}


In [6]:
documents = []

supplier_names = ['Magna', 'Bosch', 'Denso', 'Lear', 'Aptiv']
plants = ['PLT100', 'PLT200', 'PLT300', 'PLT400']
issue_types = ['weld defect', 'paint inconsistency', 'sensor failure', 'material shortage', 'assembly variance']
equipment_ids = ['EQP1001', 'EQP1002', 'EQP1003', 'EQP1004']

for i in range(1, 501):
    doc_type = random.choice(project_schema['document_types'])
    plant = random.choice(plants)
    supplier = random.choice(supplier_names)
    issue = random.choice(issue_types)
    equipment = random.choice(equipment_ids)
    amount = max(1000.0, round(float(np.random.normal(12500, 3500)), 2))
    doc_date = f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    change_order_id = f"ECO-{random.randint(10000,99999)}"

    if doc_type == 'quality_report':
        domain = 'quality'
        text = (
            f"Document ID DOC{i:05d}. Domain {domain}. Plant {plant}. "
            f"Quality inspection found {issue} on equipment {equipment}. "
            f"Document date {doc_date}. Supplier reference {supplier}. "
            f"The report describes defect observations, severity review, corrective action planning, "
            f"and quality impact for manufacturing operations."
        )
    elif doc_type == 'supplier_invoice':
        domain = 'supplier'
        text = (
            f"Document ID DOC{i:05d}. Domain {domain}. Supplier invoice from {supplier} for plant {plant}. "
            f"Invoice amount ${amount}. Document date {doc_date}. Material issue related to {issue}. "
            f"The invoice includes supplier billing context, operational dependencies, and cost review notes."
        )
    elif doc_type == 'maintenance_record':
        domain = 'maintenance'
        text = (
            f"Document ID DOC{i:05d}. Domain {domain}. Maintenance record for equipment {equipment} at plant {plant}. "
            f"Reported issue {issue}. Service date {doc_date}. Vendor {supplier}. "
            f"The maintenance note explains equipment condition, service actions, operational risk, "
            f"and recommended follow-up activities."
        )
    else:
        domain = 'engineering'
        text = (
            f"Document ID DOC{i:05d}. Domain {domain}. Engineering change notice {change_order_id} for plant {plant}. "
            f"Change related to {issue}. Effective date {doc_date}. Supplier {supplier}. "
            f"The change notice explains engineering context, downstream impact, review requirements, "
            f"and implementation details for production teams."
        )

    documents.append({
        'document_id': f'DOC{i:05d}',
        'domain': domain,
        'document_type': doc_type,
        'document_text': text
    })

documents_df = pd.DataFrame(documents)
documents_df.to_csv(RAW_PATH / 'enterprise_documents.csv', index=False)

print(documents_df.head())
print('Total documents:', len(documents_df))

  document_id   domain   document_type  \
0    DOC00001  quality  quality_report   
1    DOC00002  quality  quality_report   
2    DOC00003  quality  quality_report   
3    DOC00004  quality  quality_report   
4    DOC00005  quality  quality_report   

                                       document_text  
0  Document ID DOC00001. Domain quality. Plant PL...  
1  Document ID DOC00002. Domain quality. Plant PL...  
2  Document ID DOC00003. Domain quality. Plant PL...  
3  Document ID DOC00004. Domain quality. Plant PL...  
4  Document ID DOC00005. Domain quality. Plant PL...  
Total documents: 500


In [7]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

documents_df = pd.read_csv(RAW_PATH / 'enterprise_documents.csv')
documents_df['normalized_text'] = documents_df['document_text'].apply(normalize_text)

documents_df.to_csv(PROCESSED_PATH / 'processed_documents.csv', index=False)

print('Processed documents saved successfully.')
print(documents_df[['document_id', 'document_type', 'normalized_text']].head())

Processed documents saved successfully.
  document_id   document_type  \
0    DOC00001  quality_report   
1    DOC00002  quality_report   
2    DOC00003  quality_report   
3    DOC00004  quality_report   
4    DOC00005  quality_report   

                                     normalized_text  
0  document id doc00001. domain quality. plant pl...  
1  document id doc00002. domain quality. plant pl...  
2  document id doc00003. domain quality. plant pl...  
3  document id doc00004. domain quality. plant pl...  
4  document id doc00005. domain quality. plant pl...  


In [8]:
X = documents_df['normalized_text']
y = documents_df['document_type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

classification_vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = classification_vectorizer.fit_transform(X_train)
X_test_vec = classification_vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)
preds = clf.predict(X_test_vec)

accuracy = accuracy_score(y_test, preds)

print('Classification Accuracy:', round(float(accuracy), 4))
print(classification_report(y_test, preds))

classification_results = pd.DataFrame({
    'document_text': X_test.values,
    'actual_type': y_test.values,
    'predicted_type': preds
})

classification_results.to_csv(OUTPUT_PATH / 'classification_results.csv', index=False)

Classification Accuracy: 1.0
                           precision    recall  f1-score   support

engineering_change_notice       1.00      1.00      1.00        28
       maintenance_record       1.00      1.00      1.00        25
           quality_report       1.00      1.00      1.00        22
         supplier_invoice       1.00      1.00      1.00        25

                 accuracy                           1.00       100
                macro avg       1.00      1.00      1.00       100
             weighted avg       1.00      1.00      1.00       100



In [9]:
def extract_field(pattern, text, group_index=1):
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        return match.group(group_index)
    return None

extracted_rows = []

for _, row in documents_df.iterrows():
    text = row['document_text']

    extracted_rows.append({
        'document_id': row['document_id'],
        'domain': row['domain'],
        'document_type': row['document_type'],
        'plant_code': extract_field(r'Plant\s+(PLT\d+)|plant\s+(PLT\d+)', text, 1),
        'supplier_name': extract_field(r'(?:Supplier invoice from|Supplier reference|Vendor|Supplier)\s+(Magna|Bosch|Denso|Lear|Aptiv)', text, 1),
        'invoice_amount': extract_field(r'\$(\d+\.?\d*)', text, 1),
        'equipment_id': extract_field(r'equipment\s+(EQP\d+)', text, 1),
        'change_order_id': extract_field(r'(ECO-\d+)', text, 1),
        'document_date': extract_field(r'(\d{4}-\d{2}-\d{2})', text, 1)
    })

extracted_df = pd.DataFrame(extracted_rows)

def fix_plant_code(row):
    if pd.notna(row['plant_code']):
        return row['plant_code']
    text = documents_df.loc[documents_df['document_id'] == row['document_id'], 'document_text'].iloc[0]
    m = re.search(r'Plant\s+(PLT\d+)|plant\s+(PLT\d+)', text, re.IGNORECASE)
    if m:
        return m.group(1) if m.group(1) else m.group(2)
    return None

extracted_df['plant_code'] = extracted_df.apply(fix_plant_code, axis=1)

extracted_df.to_csv(OUTPUT_PATH / 'extracted_entities.csv', index=False)

print(extracted_df.head())

  document_id   domain   document_type plant_code supplier_name  \
0    DOC00001  quality  quality_report     PLT100         Denso   
1    DOC00002  quality  quality_report     PLT400         Magna   
2    DOC00003  quality  quality_report     PLT200         Aptiv   
3    DOC00004  quality  quality_report     PLT200          Lear   
4    DOC00005  quality  quality_report     PLT100          Lear   

  invoice_amount equipment_id change_order_id document_date  
0           None      EQP1002            None    2024-03-24  
1           None      EQP1001            None    2024-04-08  
2           None      EQP1002            None    2024-08-19  
3           None      EQP1003            None    2024-03-07  
4           None      EQP1003            None    2024-06-20  


In [10]:
documents_enriched_df = documents_df.merge(
    extracted_df[
        [
            'document_id',
            'plant_code',
            'supplier_name',
            'invoice_amount',
            'equipment_id',
            'change_order_id',
            'document_date'
        ]
    ],
    on='document_id',
    how='left'
)

documents_enriched_df.to_csv(PROCESSED_PATH / 'documents_enriched.csv', index=False)

print(documents_enriched_df.head())

  document_id   domain   document_type  \
0    DOC00001  quality  quality_report   
1    DOC00002  quality  quality_report   
2    DOC00003  quality  quality_report   
3    DOC00004  quality  quality_report   
4    DOC00005  quality  quality_report   

                                       document_text  \
0  Document ID DOC00001. Domain quality. Plant PL...   
1  Document ID DOC00002. Domain quality. Plant PL...   
2  Document ID DOC00003. Domain quality. Plant PL...   
3  Document ID DOC00004. Domain quality. Plant PL...   
4  Document ID DOC00005. Domain quality. Plant PL...   

                                     normalized_text plant_code supplier_name  \
0  document id doc00001. domain quality. plant pl...     PLT100         Denso   
1  document id doc00002. domain quality. plant pl...     PLT400         Magna   
2  document id doc00003. domain quality. plant pl...     PLT200         Aptiv   
3  document id doc00004. domain quality. plant pl...     PLT200          Lear   
4  do

In [11]:
def simple_chunk_text(text, chunk_size=30):
    words = str(text).split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

chunk_rows = []

for _, row in documents_enriched_df.iterrows():
    chunks = simple_chunk_text(row['normalized_text'], chunk_size=30)

    for idx, chunk in enumerate(chunks, start=1):
        chunk_rows.append({
            'document_id': row['document_id'],
            'chunk_id': f"{row['document_id']}_CHUNK_{idx}",
            'domain': row['domain'],
            'document_type': row['document_type'],
            'plant_code': row['plant_code'],
            'supplier_name': row['supplier_name'],
            'invoice_amount': row['invoice_amount'],
            'equipment_id': row['equipment_id'],
            'change_order_id': row['change_order_id'],
            'document_date': row['document_date'],
            'chunk_text': chunk
        })

chunk_df = pd.DataFrame(chunk_rows)
chunk_df.to_csv(CHUNK_PATH / 'chunked_documents.csv', index=False)

print(chunk_df.head())
print('Total chunks:', len(chunk_df))

  document_id          chunk_id   domain   document_type plant_code  \
0    DOC00001  DOC00001_CHUNK_1  quality  quality_report     PLT100   
1    DOC00001  DOC00001_CHUNK_2  quality  quality_report     PLT100   
2    DOC00002  DOC00002_CHUNK_1  quality  quality_report     PLT400   
3    DOC00002  DOC00002_CHUNK_2  quality  quality_report     PLT400   
4    DOC00003  DOC00003_CHUNK_1  quality  quality_report     PLT200   

  supplier_name invoice_amount equipment_id change_order_id document_date  \
0         Denso           None      EQP1002            None    2024-03-24   
1         Denso           None      EQP1002            None    2024-03-24   
2         Magna           None      EQP1001            None    2024-04-08   
3         Magna           None      EQP1001            None    2024-04-08   
4         Aptiv           None      EQP1002            None    2024-08-19   

                                          chunk_text  
0  document id doc00001. domain quality. plant pl...  


In [12]:
chunk_df['vector_ready_text'] = (
    'Domain: ' + chunk_df['domain'].fillna('UNKNOWN').astype(str) + ' | ' +
    'DocumentType: ' + chunk_df['document_type'].fillna('UNKNOWN').astype(str) + ' | ' +
    'Plant: ' + chunk_df['plant_code'].fillna('UNKNOWN').astype(str) + ' | ' +
    'Supplier: ' + chunk_df['supplier_name'].fillna('UNKNOWN').astype(str) + ' | ' +
    'Equipment: ' + chunk_df['equipment_id'].fillna('UNKNOWN').astype(str) + ' | ' +
    'ChangeOrder: ' + chunk_df['change_order_id'].fillna('UNKNOWN').astype(str) + ' | ' +
    'Date: ' + chunk_df['document_date'].fillna('UNKNOWN').astype(str) + ' | ' +
    'Text: ' + chunk_df['chunk_text'].fillna('').astype(str)
)

chunk_df.to_csv(VECTOR_PATH / 'vector_ready_records.csv', index=False)
chunk_df.to_csv(OUTPUT_PATH / 'vector_ready_records.csv', index=False)

print(chunk_df[['chunk_id', 'vector_ready_text']].head())

           chunk_id                                  vector_ready_text
0  DOC00001_CHUNK_1  Domain: quality | DocumentType: quality_report...
1  DOC00001_CHUNK_2  Domain: quality | DocumentType: quality_report...
2  DOC00002_CHUNK_1  Domain: quality | DocumentType: quality_report...
3  DOC00002_CHUNK_2  Domain: quality | DocumentType: quality_report...
4  DOC00003_CHUNK_1  Domain: quality | DocumentType: quality_report...


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

retrieval_vectorizer = TfidfVectorizer(stop_words='english')
retrieval_matrix = retrieval_vectorizer.fit_transform(chunk_df['vector_ready_text'])

print('Retrieval matrix shape:', retrieval_matrix.shape)

Retrieval matrix shape: (1000, 918)


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: build retrieval index
retrieval_vectorizer = TfidfVectorizer(stop_words='english')
retrieval_matrix = retrieval_vectorizer.fit_transform(chunk_df['vector_ready_text'])

print('Retrieval matrix shape:', retrieval_matrix.shape)

# Step 2: run query
query = 'What quality issue was reported for equipment EQP1002 in plant PLT200 and what business impact was mentioned?'

query_vector = retrieval_vectorizer.transform([query])
similarity_scores = cosine_similarity(query_vector, retrieval_matrix).flatten()

top_k = 5
top_indices = similarity_scores.argsort()[-top_k:][::-1]

retrieval_results_df = chunk_df.iloc[top_indices].copy()
retrieval_results_df['similarity_score'] = similarity_scores[top_indices]

retrieval_results_df = retrieval_results_df[
    [
        'chunk_id',
        'document_id',
        'domain',
        'document_type',
        'plant_code',
        'supplier_name',
        'equipment_id',
        'change_order_id',
        'document_date',
        'chunk_text',
        'similarity_score'
    ]
].reset_index(drop=True)

retrieval_results_df.to_csv(RETRIEVAL_PATH / 'top_k_retrieval_results.csv', index=False)
retrieval_results_df.to_csv(OUTPUT_PATH / 'top_k_retrieval_results.csv', index=False)

print(retrieval_results_df.head(top_k))

Retrieval matrix shape: (1000, 918)
           chunk_id document_id   domain   document_type plant_code  \
0  DOC00330_CHUNK_2    DOC00330  quality  quality_report     PLT200   
1  DOC00364_CHUNK_2    DOC00364  quality  quality_report     PLT200   
2  DOC00298_CHUNK_2    DOC00298  quality  quality_report     PLT200   
3  DOC00041_CHUNK_2    DOC00041  quality  quality_report     PLT200   
4  DOC00360_CHUNK_2    DOC00360  quality  quality_report     PLT200   

  supplier_name equipment_id change_order_id document_date  \
0         Denso      EQP1002            None    2024-10-05   
1         Denso      EQP1002            None    2024-08-04   
2         Aptiv      EQP1002            None    2024-12-08   
3         Aptiv      EQP1002            None    2024-06-03   
4         Bosch      EQP1002            None    2024-09-06   

                                          chunk_text  similarity_score  
0  planning, and quality impact for manufacturing...          0.488292  
1  planning, and q

In [16]:
query = 'What quality issue was reported for equipment EQP1002 in plant PLT200 and what business impact was mentioned?'

query_vector = retrieval_vectorizer.transform([query])
similarity_scores = cosine_similarity(query_vector, retrieval_matrix).flatten()

top_k = 5
top_indices = similarity_scores.argsort()[-top_k:][::-1]

retrieval_results_df = chunk_df.iloc[top_indices].copy()
retrieval_results_df['similarity_score'] = similarity_scores[top_indices]
retrieval_results_df = retrieval_results_df[
    [
        'chunk_id',
        'document_id',
        'domain',
        'document_type',
        'plant_code',
        'supplier_name',
        'equipment_id',
        'change_order_id',
        'document_date',
        'chunk_text',
        'similarity_score'
    ]
].reset_index(drop=True)

retrieval_results_df.to_csv(RETRIEVAL_PATH / 'top_k_retrieval_results.csv', index=False)
retrieval_results_df.to_csv(OUTPUT_PATH / 'top_k_retrieval_results.csv', index=False)

retrieval_results_df.head(top_k)

,chunk_id,document_id,domain,document_type,plant_code,supplier_name,equipment_id,change_order_id,document_date,chunk_text,similarity_score
0,DOC00330_CHUNK_2,DOC00330,quality,quality_report,PLT200,Denso,EQP1002,None,2024-10-05,"planning, and quality impact for manufacturing...",0.488292
1,DOC00364_CHUNK_2,DOC00364,quality,quality_report,PLT200,Denso,EQP1002,None,2024-08-04,"planning, and quality impact for manufacturing...",0.487797
2,DOC00298_CHUNK_2,DOC00298,quality,quality_report,PLT200,Aptiv,EQP1002,None,2024-12-08,"planning, and quality impact for manufacturing...",0.487241
3,DOC00041_CHUNK_2,DOC00041,quality,quality_report,PLT200,Aptiv,EQP1002,None,2024-06-03,"planning, and quality impact for manufacturing...",0.486713
4,DOC00360_CHUNK_2,DOC00360,quality,quality_report,PLT200,Bosch,EQP1002,None,2024-09-06,"planning, and quality impact for manufacturing...",0.485606


In [17]:
def build_context(results_df, top_n=3):
    selected = results_df.head(top_n)
    context_parts = []

    for i, row in selected.iterrows():
        part = (
            f"Context {i+1}: "
            f"Document {row['document_id']} | "
            f"Type {row['document_type']} | "
            f"Plant {row['plant_code']} | "
            f"Supplier {row['supplier_name']} | "
            f"Equipment {row['equipment_id']} | "
            f"Date {row['document_date']} | "
            f"Text: {row['chunk_text']}"
        )
        context_parts.append(part)

    return '\n'.join(context_parts)

grounded_context = build_context(retrieval_results_df, top_n=3)

with open(OUTPUT_PATH / 'grounded_context.txt', 'w') as f:
    f.write(grounded_context)

print(grounded_context)

Context 1: Document DOC00330 | Type quality_report | Plant PLT200 | Supplier Denso | Equipment EQP1002 | Date 2024-10-05 | Text: planning, and quality impact for manufacturing operations.
Context 2: Document DOC00364 | Type quality_report | Plant PLT200 | Supplier Denso | Equipment EQP1002 | Date 2024-08-04 | Text: planning, and quality impact for manufacturing operations.
Context 3: Document DOC00298 | Type quality_report | Plant PLT200 | Supplier Aptiv | Equipment EQP1002 | Date 2024-12-08 | Text: planning, and quality impact for manufacturing operations.


In [18]:
def first_non_null(series, default='UNKNOWN'):
    valid = series.dropna()
    if len(valid) > 0:
        return str(valid.iloc[0])
    return default

def generate_grounded_response(query, results_df):
    top_rows = results_df.head(3)

    top_document_ids = ', '.join(top_rows['document_id'].astype(str).tolist())
    document_type = first_non_null(top_rows['document_type'])
    plant_code = first_non_null(top_rows['plant_code'])
    equipment_id = first_non_null(top_rows['equipment_id'])
    supplier_name = first_non_null(top_rows['supplier_name'])
    document_date = first_non_null(top_rows['document_date'])

    issue_matches = []
    for txt in top_rows['chunk_text'].astype(str):
        m = re.search(
            r'(weld defect|paint inconsistency|sensor failure|material shortage|assembly variance)',
            txt,
            re.IGNORECASE
        )
        if m:
            issue_matches.append(m.group(1).lower())

    issue_type = issue_matches[0] if len(issue_matches) > 0 else 'not clearly identified'

    evidence_lines = []
    for _, row in top_rows.iterrows():
        evidence_lines.append(
            f"- {row['document_id']} ({row['document_type']}, score={row['similarity_score']:.4f}): {row['chunk_text']}"
        )

    answer = f"""
Question:
{query}

Grounded Answer:
Based on the top retrieved enterprise document chunks, the most relevant issue appears to be "{issue_type}" for equipment {equipment_id} in plant {plant_code}.
The retrieved context points mainly to document type "{document_type}" and references supplier "{supplier_name}" with document date "{document_date}".
The answer is grounded in retrieved enterprise records rather than model-only memory, which improves contextual reasoning and reduces unsupported responses.

Top Supporting Documents:
{top_document_ids}

Evidence:
{chr(10).join(evidence_lines)}
""".strip()

    return answer

grounded_response = generate_grounded_response(query, retrieval_results_df)

with open(OUTPUT_PATH / 'grounded_rag_response.txt', 'w') as f:
    f.write(grounded_response)

print(grounded_response)

Question:
What quality issue was reported for equipment EQP1002 in plant PLT200 and what business impact was mentioned?

Grounded Answer:
Based on the top retrieved enterprise document chunks, the most relevant issue appears to be "not clearly identified" for equipment EQP1002 in plant PLT200. 
The retrieved context points mainly to document type "quality_report" and references supplier "Denso" with document date "2024-10-05". 
The answer is grounded in retrieved enterprise records rather than model-only memory, which improves contextual reasoning and reduces unsupported responses.

Top Supporting Documents:
DOC00330, DOC00364, DOC00298

Evidence:
- DOC00330 (quality_report, score=0.4883): planning, and quality impact for manufacturing operations.
- DOC00364 (quality_report, score=0.4878): planning, and quality impact for manufacturing operations.
- DOC00298 (quality_report, score=0.4872): planning, and quality impact for manufacturing operations.


In [19]:
def tokenize_text(text):
    text = str(text).lower()
    tokens = re.findall(r'\b[a-z0-9\-]+\b', text)
    return set(tokens)

def compute_context_coverage(query, context):
    q_tokens = tokenize_text(query)
    c_tokens = tokenize_text(context)
    if len(q_tokens) == 0:
        return 0.0
    return round(len(q_tokens.intersection(c_tokens)) / len(q_tokens), 4)

def compute_groundedness(response, context):
    r_tokens = tokenize_text(response)
    c_tokens = tokenize_text(context)
    if len(r_tokens) == 0:
        return 0.0
    return round(len(r_tokens.intersection(c_tokens)) / len(r_tokens), 4)

def compute_average_retrieval_score(results_df):
    if len(results_df) == 0:
        return 0.0
    return round(float(results_df['similarity_score'].mean()), 4)

context_coverage = compute_context_coverage(query, grounded_context)
groundedness_score = compute_groundedness(grounded_response, grounded_context)
average_retrieval_score = compute_average_retrieval_score(retrieval_results_df)

evaluation_summary = {
    'query': query,
    'top_k': top_k,
    'context_coverage': context_coverage,
    'groundedness_score': groundedness_score,
    'average_retrieval_score': average_retrieval_score
}

with open(OUTPUT_PATH / 'rag_evaluation_summary.json', 'w') as f:
    json.dump(evaluation_summary, f, indent=2)

print(json.dumps(evaluation_summary, indent=2))

{
  "query": "What quality issue was reported for equipment EQP1002 in plant PLT200 and what business impact was mentioned?",
  "top_k": 5,
  "context_coverage": 0.5333,
  "groundedness_score": 0.2958,
  "average_retrieval_score": 0.4871
}


In [20]:
project_summary = {
    'generated_at': datetime.now().isoformat(),
    'raw_document_count': int(len(documents_df)),
    'processed_document_count': int(len(documents_enriched_df)),
    'classification_accuracy': round(float(accuracy), 4),
    'extracted_record_count': int(len(extracted_df)),
    'chunk_count': int(len(chunk_df)),
    'vector_ready_record_count': int(len(chunk_df)),
    'retrieved_result_count': int(len(retrieval_results_df)),
    'sample_query': query,
    'context_coverage': context_coverage,
    'groundedness_score': groundedness_score,
    'average_retrieval_score': average_retrieval_score
}

with open(OUTPUT_PATH / 'project_summary.json', 'w') as f:
    json.dump(project_summary, f, indent=2)

print(json.dumps(project_summary, indent=2))

{
  "generated_at": "2026-03-17T19:45:03.130216",
  "raw_document_count": 500,
  "processed_document_count": 500,
  "classification_accuracy": 1.0,
  "extracted_record_count": 500,
  "chunk_count": 1000,
  "vector_ready_record_count": 1000,
  "retrieved_result_count": 5,
  "sample_query": "What quality issue was reported for equipment EQP1002 in plant PLT200 and what business impact was mentioned?",
  "context_coverage": 0.5333,
  "groundedness_score": 0.2958,
  "average_retrieval_score": 0.4871
}


In [21]:
print('--- TOP RETRIEVAL RESULTS ---')
display(retrieval_results_df.head(5))

print('\n--- GROUNDED RESPONSE ---')
print(grounded_response)

print('\n--- PROJECT SUMMARY ---')
print(json.dumps(project_summary, indent=2))

--- TOP RETRIEVAL RESULTS ---


,chunk_id,document_id,domain,document_type,plant_code,supplier_name,equipment_id,change_order_id,document_date,chunk_text,similarity_score
0,DOC00330_CHUNK_2,DOC00330,quality,quality_report,PLT200,Denso,EQP1002,None,2024-10-05,"planning, and quality impact for manufacturing...",0.488292
1,DOC00364_CHUNK_2,DOC00364,quality,quality_report,PLT200,Denso,EQP1002,None,2024-08-04,"planning, and quality impact for manufacturing...",0.487797
2,DOC00298_CHUNK_2,DOC00298,quality,quality_report,PLT200,Aptiv,EQP1002,None,2024-12-08,"planning, and quality impact for manufacturing...",0.487241
3,DOC00041_CHUNK_2,DOC00041,quality,quality_report,PLT200,Aptiv,EQP1002,None,2024-06-03,"planning, and quality impact for manufacturing...",0.486713
4,DOC00360_CHUNK_2,DOC00360,quality,quality_report,PLT200,Bosch,EQP1002,None,2024-09-06,"planning, and quality impact for manufacturing...",0.485606



--- GROUNDED RESPONSE ---
Question:
What quality issue was reported for equipment EQP1002 in plant PLT200 and what business impact was mentioned?

Grounded Answer:
Based on the top retrieved enterprise document chunks, the most relevant issue appears to be "not clearly identified" for equipment EQP1002 in plant PLT200. 
The retrieved context points mainly to document type "quality_report" and references supplier "Denso" with document date "2024-10-05". 
The answer is grounded in retrieved enterprise records rather than model-only memory, which improves contextual reasoning and reduces unsupported responses.

Top Supporting Documents:
DOC00330, DOC00364, DOC00298

Evidence:
- DOC00330 (quality_report, score=0.4883): planning, and quality impact for manufacturing operations.
- DOC00364 (quality_report, score=0.4878): planning, and quality impact for manufacturing operations.
- DOC00298 (quality_report, score=0.4872): planning, and quality impact for manufacturing operations.

--- PROJEC